In [63]:
from google.cloud import bigquery
from extract_from_netsuite import pull_data_by_sql
import pandas as pd
from decimal import Decimal
import queue, concurrent.futures
import traceback

## ETL for classification table

In [ ]:
q = """
Select 
 id,
 name, 
 parent,
 fullname,
 isinactive, 
 subsidiary,
 lastmodifieddate
From classification
"""
classification = pull_data_by_sql(query=q, return_df=True)

In [ ]:
df = classification.copy()
df["id"] = pd.to_numeric(df["id"], errors="raise").astype("Int64")
df["name"] = df["name"].astype("string")
df["parent"] = pd.to_numeric(df["parent"], errors="coerce").astype("Int64")
df["fullname"] = df["fullname"].astype("string")
df["subsidiary"] = df["subsidiary"].astype("string")  # keep as string in case of multi-value IDs

# NetSuite booleans commonly come back as "T"/"F"
df["isinactive"] = df["isinactive"].map({"T": True, "F": False, True: True, False: False})
# NetSuite date-time strings -> proper timestamp
df["lastmodifieddate"] = pd.to_datetime(df["lastmodifieddate"]).dt.date
df["_loaded_at"] = pd.Timestamp.now(tz="UTC")
SCHEMA = [
    bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("parent", "INTEGER"),
    bigquery.SchemaField("fullname", "STRING"),
    bigquery.SchemaField("isinactive", "BOOLEAN"),
    bigquery.SchemaField("subsidiary", "STRING"),
    bigquery.SchemaField("lastmodifieddate", "DATE"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP")
]



In [ ]:
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_ID = "classification"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
client = bigquery.Client(project= "clean-pilot-456915-t0")
job_config = bigquery.LoadJobConfig(
        schema=SCHEMA,
        write_disposition="WRITE_TRUNCATE",   # wipes+reloads — fine even when table doesn't exist yet
        create_disposition="CREATE_IF_NEEDED", # creates the table on first run
    )

job = client.load_table_from_dataframe(df, TABLE_REF, job_config=job_config)
job.result()

table = client.get_table(TABLE_REF)
print(f"Loaded {job.output_rows} rows into {TABLE_REF} (table now has {table.num_rows} rows)")

## ETL for subsidiary table

In [ ]:
q = """
Select 
	id, 
	name,
	fullname,
	parent,
	legalname,
	custrecordcustom_practice_code,
	isinactive,
	dropdownstate,
	mainaddress,
	BUILTIN.DF(mainaddress) as mainaddress_text,
	lastmodifieddate
From subsidiary
"""
subsidiary = pull_data_by_sql(query=q, return_df=True)

In [ ]:
df = subsidiary.copy()
df["id"] = pd.to_numeric(df["id"], errors="raise").astype("Int64")
df["parent"] = pd.to_numeric(df["parent"], errors="coerce").astype("Int64")
df["mainaddress"] = pd.to_numeric(df["mainaddress"], errors="coerce").astype("Int64")
df["name"] = df["name"].astype("string")
df["fullname"] = df["fullname"].astype("string")
df["legalname"] = df["legalname"].astype("string")
df["custrecordcustom_practice_code"] = df["custrecordcustom_practice_code"].astype("string")
df["dropdownstate"] = df["dropdownstate"].astype("string")
df["mainaddress_text"] = df["mainaddress_text"].astype("string")
df["isinactive"] = df["isinactive"].map({"T": True, "F": False, True: True, False: False})
df["lastmodifieddate"] = pd.to_datetime(df["lastmodifieddate"]).dt.date

df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

SCHEMA = [
    bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("fullname", "STRING"),
    bigquery.SchemaField("parent", "INTEGER"),
    bigquery.SchemaField("legalname", "STRING"),
    bigquery.SchemaField("custrecordcustom_practice_code", "STRING"),
    bigquery.SchemaField("isinactive", "BOOLEAN"),
    bigquery.SchemaField("dropdownstate", "STRING"),
    bigquery.SchemaField("mainaddress", "INTEGER"),
    bigquery.SchemaField("mainaddress_text", "STRING"),
    bigquery.SchemaField("lastmodifieddate", "DATE"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP"),
]

In [ ]:
TABLE_ID = "subsidiary"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
client = bigquery.Client(project=PROJECT_ID)
job_config = bigquery.LoadJobConfig(
    schema=SCHEMA,
    write_disposition="WRITE_TRUNCATE",
    create_disposition="CREATE_IF_NEEDED",
)

job = client.load_table_from_dataframe(df, TABLE_REF, job_config=job_config)
job.result()

table = client.get_table(TABLE_REF)
print(f"Loaded {job.output_rows} rows into {TABLE_REF} (table now has {table.num_rows} rows)")

## ETL for table - item

In [ ]:
q = """
Select 
	id,
	fullname,
	itemtype,
	upccode,
	mpn,
	displayname,
	vendorname,
	purchaseunit,
	parent,
	class,
	subsidiary,
	custitemcustitem_dnd_brand,
	manufacturer,
	averagecost,
	cost,
	lastpurchaseprice,
	costingmethoddisplay,
	autoLeadTime,
	autoReorderPoint,
	autoPreferredStockLevel,
	lastmodifieddate
From item
Order by id, itemtype
"""
item = pull_data_by_sql(query=q, return_df=True)

In [ ]:
df = item.copy()

df["id"] = pd.to_numeric(df["id"], errors="raise").astype("Int64")

int_cols = ["purchaseunit", "parent", "class"]
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

string_cols = [
    "fullname", "itemtype", "upccode", "mpn", "displayname", "vendorname",
    "subsidiary", "custitemcustitem_dnd_brand", "manufacturer", "costingmethoddisplay",
]
for col in string_cols:
    df[col] = df[col].astype("string")

for col in ["averagecost", "cost", "lastpurchaseprice"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].apply(lambda x: Decimal(str(round(x, 2))) if pd.notna(x) else None)
    
for col in ["autoleadtime", "autoreorderpoint", "autopreferredstocklevel"]:
    df[col] = df[col].map({"T": True, "F": False, True: True, False: False})

df["lastmodifieddate"] = pd.to_datetime(df["lastmodifieddate"]).dt.date

df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

SCHEMA = [
bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
bigquery.SchemaField("fullname", "STRING"),
bigquery.SchemaField("itemtype", "STRING"),
bigquery.SchemaField("upccode", "STRING"),
bigquery.SchemaField("mpn", "STRING"),
bigquery.SchemaField("displayname", "STRING"),
bigquery.SchemaField("vendorname", "STRING"),
bigquery.SchemaField("purchaseunit", "INTEGER"),
bigquery.SchemaField("parent", "INTEGER"),
bigquery.SchemaField("class", "INTEGER"),
bigquery.SchemaField("subsidiary", "STRING"),
bigquery.SchemaField("custitemcustitem_dnd_brand", "STRING"),
bigquery.SchemaField("manufacturer", "STRING"),
bigquery.SchemaField("averagecost", "NUMERIC"),
bigquery.SchemaField("cost", "NUMERIC"),
bigquery.SchemaField("lastpurchaseprice", "NUMERIC"),
bigquery.SchemaField("costingmethoddisplay", "STRING"),
bigquery.SchemaField("autoleadtime", "BOOLEAN"),
bigquery.SchemaField("autoreorderpoint", "BOOLEAN"),
bigquery.SchemaField("autopreferredstocklevel", "BOOLEAN"),
bigquery.SchemaField("lastmodifieddate", "DATE"),
bigquery.SchemaField("_loaded_at", "TIMESTAMP"),
]

TABLE_ID = "item"
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
client = bigquery.Client(project=PROJECT_ID)
job_config = bigquery.LoadJobConfig(
        schema=SCHEMA,
        write_disposition="WRITE_TRUNCATE",
        create_disposition="CREATE_IF_NEEDED",
    )

job = client.load_table_from_dataframe(df, TABLE_REF, job_config=job_config)
job.result()

table = client.get_table(TABLE_REF)
print(f"Loaded {job.output_rows} rows into {TABLE_REF} (table now has {table.num_rows} rows)")

## ETL for table - customer

In [ ]:
q = """
Select 
id,
fullname,
category,
currency,
email,
entityid, 
entitystatus, 
isinactive,
entitytitle,
datecreated,
lastmodifieddate
FROM customer
"""
customer = pull_data_by_sql(query=q, return_df=True)

In [ ]:
df = customer.copy()

df["id"] = pd.to_numeric(df["id"], errors="raise").astype("Int64")

int_cols = ["category", "currency", "entitystatus"]
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

string_cols = ["email", "entityid", "entitytitle", "fullname"]
for col in string_cols:
    df[col] = df[col].astype("string")

df["isinactive"] = df["isinactive"].map({"T": True, "F": False, True: True, False: False})

df["datecreated"] = pd.to_datetime(df["datecreated"]).dt.date
df["lastmodifieddate"] = pd.to_datetime(df["lastmodifieddate"]).dt.date

df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

SCHEMA = [
    bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("category", "INTEGER"),
    bigquery.SchemaField("currency", "INTEGER"),
    bigquery.SchemaField("datecreated", "DATE"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("entityid", "STRING"),
    bigquery.SchemaField("entitystatus", "INTEGER"),
    bigquery.SchemaField("isinactive", "BOOLEAN"),
    bigquery.SchemaField("entitytitle", "STRING"),
    bigquery.SchemaField("fullname", "STRING"),
    bigquery.SchemaField("lastmodifieddate", "DATE"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP"),
]
TABLE_ID = "customer"
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
client = bigquery.Client(project=PROJECT_ID)
job_config = bigquery.LoadJobConfig(
    schema=SCHEMA,
    write_disposition="WRITE_TRUNCATE",
    create_disposition="CREATE_IF_NEEDED",
)

job = client.load_table_from_dataframe(df, TABLE_REF, job_config=job_config)
job.result()

table = client.get_table(TABLE_REF)
print(f"Loaded {job.output_rows} rows into {TABLE_REF} (table now has {table.num_rows} rows)")

In [ ]:
from google.cloud import bigquery
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
client = bigquery.Client(project = PROJECT_ID)
tables = client.list_tables(DATASET_ID)

for table in tables:
    print(table.table_id)


## ETL for table - transactionstatus

In [69]:
ROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_ID = "transactionstatus"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

q = """
Select 
    id,
    trantype,
    name,
    fullname,
    friendlykey
From transactionstatus
"""
transactionstatus = pull_data_by_sql(query=q, return_df=True)

df = transactionstatus.copy()

string_cols = ["id", "trantype", "name", "fullname", "friendlykey"]
for col in string_cols:
    if col not in df.columns:
        df[col] = None
    df[col] = df[col].astype("string")

df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

SCHEMA = [
    bigquery.SchemaField("id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("trantype", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("fullname", "STRING"),
    bigquery.SchemaField("friendlykey", "STRING"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP"),
]

client = bigquery.Client(project=PROJECT_ID)
job_config = bigquery.LoadJobConfig(
    schema=SCHEMA,
    write_disposition="WRITE_TRUNCATE",
    create_disposition="CREATE_IF_NEEDED",
)
job = client.load_table_from_dataframe(df, TABLE_REF, job_config=job_config)
job.result()

table = client.get_table(TABLE_REF)
print(f"Loaded {job.output_rows} rows into {TABLE_REF} (table now has {table.num_rows} rows)")

Total results: 324
Final count: 324 | Total time: 1.25s


c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 324 rows into clean-pilot-456915-t0.NetSuite.transactionstatus (table now has 324 rows)


## ETL for table - transaction

In [66]:
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_ID = "transaction"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

def cast_transaction_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["id"] = pd.to_numeric(df["id"], errors="raise").astype("Int64")

    int_cols = ["entity", "employee"]
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    string_cols = ["tranid", "transactionnumber", "status", "type"]
    for col in string_cols:
        df[col] = df[col].astype("string")

    date_cols = ["trandate", "createddate", "closedate", "lastmodifieddate"]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce").dt.date

    df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

    return df

SCHEMA = [
    bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("tranid", "STRING"),
    bigquery.SchemaField("transactionnumber", "STRING"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("type", "STRING"),
    bigquery.SchemaField("trandate", "DATE"),
    bigquery.SchemaField("createddate", "DATE"),
    bigquery.SchemaField("closedate", "DATE"),
    bigquery.SchemaField("entity", "INTEGER"),
    bigquery.SchemaField("employee", "INTEGER"),
    bigquery.SchemaField("lastmodifieddate", "DATE"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP"),
]

def backfill_transaction(chunk_size=5000):
    client = bigquery.Client(project=PROJECT_ID)
    all_chunks = []
    last_id = 0

    while True:
        query = f"""
            SELECT id, tranid, transactionnumber, status, type, entity, employee,
                   tranDate, createdDate, closeDate, lastmodifieddate
            FROM transaction
            WHERE id > {last_id}
            ORDER BY id
            FETCH NEXT {chunk_size} ROWS ONLY
        """
        df = pull_data_by_sql(query=query, return_df=True)

        if df.empty:
            print("Backfill complete.")
            break

        all_chunks.append(df)
        last_id = int(df["id"].astype("int64").max())
        print(f"Pulled up to id {last_id} ({len(df)} rows this page)")

        if len(df) < chunk_size:
            print("Backfill complete.")
            break

    full_df = pd.concat(all_chunks, ignore_index=True)
    full_df = cast_transaction_df(full_df)

    job_config = bigquery.LoadJobConfig(
        schema=SCHEMA,
        write_disposition="WRITE_TRUNCATE",
        create_disposition="CREATE_IF_NEEDED",
        time_partitioning=bigquery.TimePartitioning(type_=bigquery.TimePartitioningType.MONTH, field="trandate"),
        clustering_fields=["type"],
    )
    job = client.load_table_from_dataframe(full_df, TABLE_REF, job_config=job_config)
    job.result()

    print(f"Loaded {job.output_rows} rows into {TABLE_REF}")

In [68]:

backfill_transaction()

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Total results: 5000
Fetched: 2000 | Page time: 1.85s
Fetched: 3000 | Page time: 1.79s
Fetched: 4000 | Page time: 1.80s
Fetched: 5000 | Page time: 1.74s
Final count: 5000 | Total time: 9.60s
Pulled up to id 9178 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 1.92s
Fetched: 3000 | Page time: 1.86s
Fetched: 4000 | Page time: 1.76s
Fetched: 5000 | Page time: 1.74s
Final count: 5000 | Total time: 10.04s
Pulled up to id 19330 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 1.91s
Fetched: 3000 | Page time: 1.87s
Fetched: 4000 | Page time: 1.81s
Fetched: 5000 | Page time: 1.80s
Final count: 5000 | Total time: 9.60s
Pulled up to id 24840 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 1.97s
Fetched: 3000 | Page time: 2.10s
Fetched: 4000 | Page time: 1.89s
Fetched: 5000 | Page time: 1.87s
Final count: 5000 | Total time: 10.24s
Pulled up to id 30313 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 1.76s
Fetched: 3

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 1272421 rows into clean-pilot-456915-t0.NetSuite.transaction


In [64]:
TABLE_REF

'clean-pilot-456915-t0.NetSuite.transaction'

In [65]:
client = bigquery.Client(project=PROJECT_ID)
client.delete_table(TABLE_REF, not_found_ok=True)

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


# ETL for table - AggregateItemLocation

In [9]:
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_ID = "aggregateItemLocation"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

def cast_aggregateitemlocation_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["item"] = pd.to_numeric(df["item"], errors="raise").astype("Int64")
    df["location"] = pd.to_numeric(df["location"], errors="coerce").astype("Int64")
    df["leadtime"] = pd.to_numeric(df["leadtime"], errors="coerce").astype("Int64")

    float_cols = [
        "quantityavailable", "quantityonhand", "quantityintransit", "quantitycommitted",
        "quantitybackordered", "quantityonorder", "averagecostmli", "lastpurchasepricemli",
        "onhandvaluemli", "reorderpoint", "safetystocklevel", "preferredstocklevel",
    ]
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["lastquantityavailablechange"] = pd.to_datetime(df["lastquantityavailablechange"], errors="coerce").dt.date
    df["lastmodifieddate"] = pd.to_datetime(df["lastmodifieddate"], errors="coerce").dt.date

    df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

    return df

SCHEMA = [
    bigquery.SchemaField("item", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("location", "INTEGER"),
    bigquery.SchemaField("quantityavailable", "FLOAT"),
    bigquery.SchemaField("quantityonhand", "FLOAT"),
    bigquery.SchemaField("quantityintransit", "FLOAT"),
    bigquery.SchemaField("quantitycommitted", "FLOAT"),
    bigquery.SchemaField("quantitybackordered", "FLOAT"),
    bigquery.SchemaField("quantityonorder", "FLOAT"),
    bigquery.SchemaField("averagecostmli", "FLOAT"),
    bigquery.SchemaField("lastpurchasepricemli", "FLOAT"),
    bigquery.SchemaField("onhandvaluemli", "FLOAT"),
    bigquery.SchemaField("reorderpoint", "FLOAT"),
    bigquery.SchemaField("safetystocklevel", "FLOAT"),
    bigquery.SchemaField("preferredstocklevel", "FLOAT"),
    bigquery.SchemaField("leadtime", "INTEGER"),
    bigquery.SchemaField("lastquantityavailablechange", "DATE"),
    bigquery.SchemaField("lastmodifieddate", "DATE"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP"),
]

def load_aggregateitemlocation(chunk_size=5000):
    client = bigquery.Client(project=PROJECT_ID)
    all_chunks = []
    last_item, last_location = 0, 0

    while True:
        query = f"""
            SELECT item, location, quantityavailable, quantityonhand, quantityintransit,
                   quantitycommitted, quantitybackordered, quantityonorder, averagecostmli,
                   lastpurchasepricemli, onhandvaluemli, reorderpoint, safetystocklevel,
                   preferredstocklevel, leadtime, lastquantityavailablechange, lastmodifieddate
            FROM aggregateItemLocation
            WHERE item > {last_item} OR (item = {last_item} AND location > {last_location})
            ORDER BY item, location
            FETCH NEXT {chunk_size} ROWS ONLY
        """
        df = pull_data_by_sql(query=query, return_df=True)

        if df.empty:
            print("Load complete.")
            break

        all_chunks.append(df)
        last_row = df.iloc[-1]
        last_item = int(last_row["item"])
        last_location = int(last_row["location"])
        print(f"Pulled through item={last_item}, location={last_location} ({len(df)} rows this page)")

        if len(df) < chunk_size:
            print("Load complete.")
            break

    full_df = pd.concat(all_chunks, ignore_index=True)
    full_df = cast_aggregateitemlocation_df(full_df)

    job_config = bigquery.LoadJobConfig(
        schema=SCHEMA,
        write_disposition="WRITE_TRUNCATE",
        create_disposition="CREATE_IF_NEEDED",
        clustering_fields=["location"]
    )
    job = client.load_table_from_dataframe(full_df, TABLE_REF, job_config=job_config)
    job.result()

    print(f"Loaded {job.output_rows} rows into {TABLE_REF}")

In [10]:
load_aggregateitemlocation()

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Total results: 5000
Fetched: 2000 | Page time: 0.71s
Fetched: 3000 | Page time: 0.68s
Fetched: 4000 | Page time: 0.79s
Fetched: 5000 | Page time: 1.11s
Final count: 5000 | Total time: 5.24s
Pulled through item=2510, location=24 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 0.99s
Fetched: 3000 | Page time: 0.71s
Fetched: 4000 | Page time: 0.68s
Fetched: 5000 | Page time: 0.72s
Final count: 5000 | Total time: 4.52s
Pulled through item=4566, location=24 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 0.74s
Fetched: 3000 | Page time: 1.24s
Fetched: 4000 | Page time: 0.75s
Fetched: 5000 | Page time: 0.73s
Final count: 5000 | Total time: 4.42s
Pulled through item=7983, location=24 (5000 rows this page)
Total results: 5000
Fetched: 2000 | Page time: 0.74s
Fetched: 3000 | Page time: 0.82s
Fetched: 4000 | Page time: 0.71s
Fetched: 5000 | Page time: 1.14s
Final count: 5000 | Total time: 4.64s
Pulled through item=8438, location=42 (5000 rows this page)


c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 348477 rows into clean-pilot-456915-t0.NetSuite.aggregateItemLocation


# ETL for table - transactionLine

In [ ]:
PROJECT_ID = "clean-pilot-456915-t0"
DATASET_ID = "NetSuite"
TABLE_ID = "transactionline"
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

SELECT_COLUMNS = """
    uniquekey, transaction, item, itemtype, accountinglinetype, expenseaccount,
    inventorylocation, netamount, costestimate, rate, price, quantity,
    quantitybackordered, createdfrom, memo, mainline, taxline,
    custcolfree_goods_checkbox, linecreateddate, linelastmodifieddate
"""
CLUSTERING_FIELDS = ["mainline", "taxline", "transaction"]

SCHEMA = [
    bigquery.SchemaField("uniquekey", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("transaction", "INTEGER"),
    bigquery.SchemaField("item", "INTEGER"),
    bigquery.SchemaField("itemtype", "STRING"),
    bigquery.SchemaField("accountinglinetype", "STRING"),
    bigquery.SchemaField("expenseaccount", "INTEGER"),
    bigquery.SchemaField("inventorylocation", "INTEGER"),
    bigquery.SchemaField("netamount", "FLOAT"),
    bigquery.SchemaField("costestimate", "FLOAT"),
    bigquery.SchemaField("rate", "FLOAT"),
    bigquery.SchemaField("price", "FLOAT"),
    bigquery.SchemaField("quantity", "FLOAT"),
    bigquery.SchemaField("quantitybackordered", "FLOAT"),
    bigquery.SchemaField("createdfrom", "INTEGER"),
    bigquery.SchemaField("memo", "STRING"),
    bigquery.SchemaField("mainline", "BOOLEAN"),
    bigquery.SchemaField("taxline", "BOOLEAN"),
    bigquery.SchemaField("custcolfree_goods_checkbox", "BOOLEAN"),
    bigquery.SchemaField("linecreateddate", "DATE"),
    bigquery.SchemaField("linelastmodifieddate", "DATE"),
    bigquery.SchemaField("_loaded_at", "TIMESTAMP")]

def cast_transactionline_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["uniquekey"] = pd.to_numeric(df["uniquekey"], errors="raise").astype("Int64")

    int_cols = ["transaction", "item", "expenseaccount", "inventorylocation", "createdfrom"]
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    string_cols = ["itemtype", "accountinglinetype", "memo"]
    for col in string_cols:
        df[col] = df[col].astype("string")

    float_cols = ["netamount", "costestimate", "rate", "price", "quantity", "quantitybackordered"]
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    bool_cols = ["mainline", "taxline", "custcolfree_goods_checkbox"]
    for col in bool_cols:
        df[col] = df[col].map({"T": True, "F": False, True: True, False: False})

    df["linecreateddate"] = pd.to_datetime(df["linecreateddate"], errors="coerce").dt.date
    df["linelastmodifieddate"] = pd.to_datetime(df["linelastmodifieddate"], errors="coerce").dt.date

    df["_loaded_at"] = pd.Timestamp.now(tz="UTC")

    return df


In [43]:
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

def create_transactionline_table():
    """Run ONCE, before the first year. Never inside a per-year run."""
    client = bigquery.Client(project=PROJECT_ID)
    client.delete_table(TABLE_REF, not_found_ok=True)
    table = bigquery.Table(TABLE_REF, schema=SCHEMA)
    table.clustering_fields = CLUSTERING_FIELDS
    client.create_table(table)
    print(f"Created {TABLE_REF}")

def _year_filter(year):
    return (f"linecreateddate >= TO_DATE('{year}-01-01','YYYY-MM-DD') "
            f"AND linecreateddate < TO_DATE('{year + 1}-01-01','YYYY-MM-DD')")

def get_year_range(year, attempts=4):
    q = f"SELECT MIN(uniquekey) as mn, MAX(uniquekey) as mx, COUNT(*) as n FROM transactionline WHERE {_year_filter(year)}"
    for _ in range(attempts):
        df = pull_data_by_sql(query=q, return_df=True)
        if not df.empty and "mn" in df.columns:
            return int(df["mn"].iloc[0]), int(df["mx"].iloc[0]), int(df["n"].iloc[0])
    raise RuntimeError(f"could not get uniquekey range for {year}")


In [52]:
def year_worker(slice_queue, year, worker_id, chunk_size=5000, batch_flush_every=4, max_slice_attempts=3):
    # per-thread work function, each thread repeatedly pulls a slice from the shared slice_queue
    client = bigquery.Client(project=PROJECT_ID)
    yf = _year_filter(year)
    total_loaded = 0
    attempts = {}

    while True:
        try:
            s_start, s_end = slice_queue.get_nowait()
        except queue.Empty:
            break

        key = (s_start, s_end)
        attempts[key] = attempts.get(key, 0) + 1
        last_uniquekey = s_start
        last_flushed = s_start
        buffer = []
        # Inside each slice
        # Queries from Netsuite
        try: 
            while True:
                query = f"""
                    SELECT {SELECT_COLUMNS}
                    FROM transactionline
                    WHERE {yf}
                      AND uniquekey > {last_uniquekey} AND uniquekey <= {s_end}
                    ORDER BY uniquekey
                    FETCH NEXT {chunk_size} ROWS ONLY
                """
                df = pull_data_by_sql(query=query, return_df=True)
                if df.empty:
                    break
                df.columns = df.columns.str.lower()

                expected_columns = [
                    "uniquekey", "transaction", "item", "itemtype",
                    "accountinglinetype", "expenseaccount", "inventorylocation",
                    "netamount", "costestimate", "rate", "price", "quantity",
                    "quantitybackordered", "createdfrom", "memo", "mainline",
                    "taxline", "custcolfree_goods_checkbox",
                    "linecreateddate", "linelastmodifieddate",
                ]

                # Add columns omitted by the source connector.
                for col in expected_columns:
                    if col not in df.columns:
                        df[col] = pd.NA

                df = df[expected_columns]
                            
              
                buffer.append(df)
                last_uniquekey = int(df["uniquekey"].astype("int64").max())

                if len(buffer) >= batch_flush_every or len(df) < chunk_size:
                    batch_df = cast_transactionline_df(pd.concat(buffer, ignore_index=True))
                    client.load_table_from_dataframe(
                        batch_df, TABLE_REF,
                        job_config=bigquery.LoadJobConfig(
                            schema=SCHEMA, write_disposition="WRITE_APPEND",
                            create_disposition="CREATE_IF_NEEDED",
                            clustering_fields=CLUSTERING_FIELDS,
                        ),
                    ).result()
                    total_loaded += len(batch_df)
                    last_flushed = last_uniquekey
                    buffer = []
                    print(f"[{year} w{worker_id}] flushed to {last_flushed} (total {total_loaded})")

                if len(df) < chunk_size:
                    break

        except Exception as e:
            if attempts[key] < max_slice_attempts and last_flushed < s_end:
                slice_queue.put((last_flushed, s_end))
                print(f"[{year} w{worker_id}] requeued ({last_flushed}, {s_end}): {e}")
            else:
                print(f"[{year} w{worker_id}] ABANDONED ({last_flushed}, {s_end}): {e}")

    return total_loaded

def load_year(year, n_workers=3, chunk_size=5000, slice_size=100000):
    mn, mx, expected = get_year_range(year)
    print(f"{year}: uniquekey {mn}..{mx}, NetSuite says {expected} rows")

    slice_queue = queue.Queue()
    start = mn - 1
    while start < mx:
        end = min(start + slice_size, mx)
        slice_queue.put((start, end))
        start = end
    print(f"{year}: {slice_queue.qsize()} slices across {n_workers} workers")

    loaded = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = [ex.submit(year_worker, slice_queue, year, i, chunk_size) for i in range(n_workers)]
        for f in concurrent.futures.as_completed(futures):
            loaded += f.result() or 0

    client = bigquery.Client(project=PROJECT_ID)
    in_bq = list(client.query(f"""
        SELECT COUNT(*) as n FROM `{TABLE_REF}`
        WHERE linecreateddate >= DATE '{year}-01-01' AND linecreateddate < DATE '{year + 1}-01-01'
    """).result())[0].n

    ok = in_bq == expected
    print(f"{year}: loaded {loaded} | BigQuery has {in_bq} | expected {expected} -> {'OK' if ok else 'MISMATCH'}")
    return ok

In [ ]:
#load_year(2026)